# NumCompute Starter Notebook

A guided tour of **NumCompute**, a NumPy-only scientific-computing toolkit (no scikit-learn, pandas or SciPy). The notebook pulls the library straight from our GitHub repository, imports it like any other library, and runs everything on the **Iris** dataset.

It follows the three things the demo has to show:

| Part | What you will see |
|------|-------------------|
| **1. Setting up and running the demo** | install/import the library from GitHub, load Iris with `io`, deal with missing values |
| **2. Using your pipeline** | `Imputer -> StandardScaler -> classifier` built with `pipeline.Pipeline`, fit on a train split, predict on a test split |
| **3. Reading outputs** | **metrics** (accuracy, precision/recall/F1, confusion matrix, ROC/AUC), **ranks** (ties, percentiles), **gradients** (finite differences, Jacobian) |

Part 4 shows the rest of the toolkit (statistics, sorting/top-k, utils) and Part 5 a loop-vs-vectorised benchmark.

### How to use this notebook
1. **Requirements:** Python 3.9+, NumPy 1.22+, Jupyter (or Google Colab), and `git` (or internet access, see the zip fallback in Step 1.2).
2. **Edit one line:** set `REPO_URL` in cell 1.1 to our GitHub repository.
3. **Run top to bottom** (`Kernel > Restart & Run All`). Every cell is safe to re-run. Total run time is about 20-40 seconds.
4. Each section has a *What to look for* note that explains how to read the printed output. Cells containing `assert` check the result against NumPy, so a clean run means the numbers were verified, not just printed.

# Part 1: Setting up and running the demo

## Step 1.1: Tell the notebook where the library lives
Only `REPO_URL` has to be edited. `BRANCH` is the branch to use (usually `main`).

In [ ]:
REPO_URL = "https://github.com/Dax555/Assignment_2.1_Programming_Task_1_-RedHats.git"
BRANCH = "main"                  # branch to download
CLONE_DIR = "numcompute_src"     # local folder the library is downloaded into

## Step 1.2: Download the library and make it importable
`setup_numcompute` does three things:
1. **Downloads** the repository with `git clone` (or, if `git` is missing, as a zip file from GitHub).
2. **Finds** the package folder inside it. It works whether the folder is called `numcompute` or `numCompute`.
3. **Adds** that location to Python's import path, so `import numcompute` works without installing anything.

> If the repository's `pyproject.toml` is filled in you can instead run `pip install "git+<REPO_URL>"` in a terminal. The steps below work either way.

In [ ]:
import getpass
import importlib
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path


def _find_package_root(base):
    """Find the folder containing the NumCompute package."""
    base = os.path.abspath(base)

    for current, dirs, files in os.walk(base):
        # Search at most two directory levels deep.
        relative = os.path.relpath(current, base)
        if relative != "." and relative.count(os.sep) >= 2:
            dirs[:] = []

        for name in ("numcompute", "numCompute"):
            package = os.path.join(current, name)
            if os.path.isfile(os.path.join(package, "stats.py")):
                return os.path.abspath(current), name

    raise FileNotFoundError(
        f"No numcompute package found inside {base!r}. "
        "Check the repository URL, branch, and package structure."
    )


def setup_numcompute(repo_url, branch="main", target="numcompute_src"):
    """Clone a private GitHub repository or download its ZIP using a PAT."""

    # Enter the token securely, without displaying it.
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        token = getpass.getpass("Enter GitHub PAT (input hidden): ")

    if not token:
        raise ValueError("A GitHub token is required.")

    repo_url = repo_url.removesuffix(".git")
    zip_url = f"{repo_url}/archive/refs/heads/{branch}.zip"

    # Git's askpass helper reads credentials from environment variables.
    askpass_script = """#!/bin/sh
case "$1" in
  *sername*) echo "x-access-token" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
"""

    try:
        if os.path.exists(target):
            shutil.rmtree(target)

        with tempfile.TemporaryDirectory() as temp_dir:
            askpass_path = os.path.join(temp_dir, "askpass.sh")

            with open(askpass_path, "w") as f:
                f.write(askpass_script)

            os.chmod(askpass_path, 0o700)

            env = os.environ.copy()
            env["GITHUB_TOKEN"] = token
            env["GIT_ASKPASS"] = askpass_path
            env["GIT_TERMINAL_PROMPT"] = "0"

            try:
                subprocess.run(
                    [
                        "git", "-c", "credential.helper=",
                        "clone", "--depth", "1",
                        "--branch", branch, repo_url, target
                    ],
                    check=True,
                    capture_output=True,
                    text=True,
                    env=env,
                )
                print("Downloaded using authenticated Git clone.")

            except (FileNotFoundError, subprocess.CalledProcessError) as exc:
                if isinstance(exc, subprocess.CalledProcessError):
                    print("Git clone failed:", exc.stderr.strip())
                else:
                    print("Git is unavailable.")

                print("Trying authenticated ZIP download...")

                request = urllib.request.Request(
                    zip_url,
                    headers={
                        "Authorization": f"Bearer {token}",
                        "User-Agent": "NumCompute-Notebook",
                    },
                )

                with urllib.request.urlopen(request) as response:
                    archive = response.read()

                os.makedirs(target, exist_ok=True)

                with zipfile.ZipFile(
                    __import__("io").BytesIO(archive)
                ) as zf:
                    # Prevent ZIP entries from writing outside target.
                    destination = Path(target).resolve()
                    for member in zf.infolist():
                        extracted = (destination / member.filename).resolve()
                        if (
                            extracted != destination
                            and destination not in extracted.parents
                        ):
                            raise ValueError(
                                "Unsafe path found in downloaded ZIP."
                            )
                    zf.extractall(target)

                print("Downloaded using authenticated ZIP.")

    except Exception:
        shutil.rmtree(target, ignore_errors=True)
        raise

    finally:
        # Avoid retaining the token in this function's local variable.
        token = None

    root, name = _find_package_root(target)

    if root not in sys.path:
        sys.path.insert(0, root)

    return name


PKG = setup_numcompute(REPO_URL, BRANCH, CLONE_DIR)
print("Package folder found:", PKG)


Enter GitHub PAT (input hidden): ··········
Downloaded using authenticated Git clone.
Package folder found: numCompute


## Step 1.3: Import the library
The library is imported module by module. `PKG` makes the next cell work for either folder spelling.

*What to look for:* the list of modules should print without errors. If you see `ModuleNotFoundError`, re-check `REPO_URL`/`BRANCH`.

In [ ]:
import warnings
import numpy as np

nc = importlib.import_module(PKG)
MODULES = ["io", "preprocessing", "sort_search", "rank", "stats", "metrics", "optim", "pipeline", "utils"]
io, preprocessing, sort_search, rank, stats, metrics, optim, pipeline, utils = (
    importlib.import_module(f"{PKG}.{m}") for m in MODULES)

np.set_printoptions(precision=4, suppress=True, linewidth=100)   # tidy array printing
SEED = 42                                                          # one seed -> identical results every run
print("python", sys.version.split()[0], "| numpy", np.__version__)
print("imported from:", os.path.dirname(nc.__file__))
print("modules:", ", ".join(MODULES))

python 3.13.16 | numpy 2.1.3
imported from: /content/numcompute_src/numCompute/numCompute
modules: io, preprocessing, sort_search, rank, stats, metrics, optim, pipeline, utils


## Step 1.4: The Iris dataset
Iris (150 flowers, 4 measurements each, 3 species) is stored inside this notebook, so it runs offline. The next cell writes it to `iris.csv`, and then we create **`iris_with_gaps.csv`**, a copy where about 5 % of the measurements are blanked out with the markers `""`, `NA` and `?`, so we can show how missing values are handled.

In [ ]:
IRIS_CSV = '''Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa
4,4.6,3.1,1.5,0.2,Iris-setosa
5,5.0,3.6,1.4,0.2,Iris-setosa
6,5.4,3.9,1.7,0.4,Iris-setosa
7,4.6,3.4,1.4,0.3,Iris-setosa
8,5.0,3.4,1.5,0.2,Iris-setosa
9,4.4,2.9,1.4,0.2,Iris-setosa
10,4.9,3.1,1.5,0.1,Iris-setosa
11,5.4,3.7,1.5,0.2,Iris-setosa
12,4.8,3.4,1.6,0.2,Iris-setosa
13,4.8,3.0,1.4,0.1,Iris-setosa
14,4.3,3.0,1.1,0.1,Iris-setosa
15,5.8,4.0,1.2,0.2,Iris-setosa
16,5.7,4.4,1.5,0.4,Iris-setosa
17,5.4,3.9,1.3,0.4,Iris-setosa
18,5.1,3.5,1.4,0.3,Iris-setosa
19,5.7,3.8,1.7,0.3,Iris-setosa
20,5.1,3.8,1.5,0.3,Iris-setosa
21,5.4,3.4,1.7,0.2,Iris-setosa
22,5.1,3.7,1.5,0.4,Iris-setosa
23,4.6,3.6,1.0,0.2,Iris-setosa
24,5.1,3.3,1.7,0.5,Iris-setosa
25,4.8,3.4,1.9,0.2,Iris-setosa
26,5.0,3.0,1.6,0.2,Iris-setosa
27,5.0,3.4,1.6,0.4,Iris-setosa
28,5.2,3.5,1.5,0.2,Iris-setosa
29,5.2,3.4,1.4,0.2,Iris-setosa
30,4.7,3.2,1.6,0.2,Iris-setosa
31,4.8,3.1,1.6,0.2,Iris-setosa
32,5.4,3.4,1.5,0.4,Iris-setosa
33,5.2,4.1,1.5,0.1,Iris-setosa
34,5.5,4.2,1.4,0.2,Iris-setosa
35,4.9,3.1,1.5,0.1,Iris-setosa
36,5.0,3.2,1.2,0.2,Iris-setosa
37,5.5,3.5,1.3,0.2,Iris-setosa
38,4.9,3.1,1.5,0.1,Iris-setosa
39,4.4,3.0,1.3,0.2,Iris-setosa
40,5.1,3.4,1.5,0.2,Iris-setosa
41,5.0,3.5,1.3,0.3,Iris-setosa
42,4.5,2.3,1.3,0.3,Iris-setosa
43,4.4,3.2,1.3,0.2,Iris-setosa
44,5.0,3.5,1.6,0.6,Iris-setosa
45,5.1,3.8,1.9,0.4,Iris-setosa
46,4.8,3.0,1.4,0.3,Iris-setosa
47,5.1,3.8,1.6,0.2,Iris-setosa
48,4.6,3.2,1.4,0.2,Iris-setosa
49,5.3,3.7,1.5,0.2,Iris-setosa
50,5.0,3.3,1.4,0.2,Iris-setosa
51,7.0,3.2,4.7,1.4,Iris-versicolor
52,6.4,3.2,4.5,1.5,Iris-versicolor
53,6.9,3.1,4.9,1.5,Iris-versicolor
54,5.5,2.3,4.0,1.3,Iris-versicolor
55,6.5,2.8,4.6,1.5,Iris-versicolor
56,5.7,2.8,4.5,1.3,Iris-versicolor
57,6.3,3.3,4.7,1.6,Iris-versicolor
58,4.9,2.4,3.3,1.0,Iris-versicolor
59,6.6,2.9,4.6,1.3,Iris-versicolor
60,5.2,2.7,3.9,1.4,Iris-versicolor
61,5.0,2.0,3.5,1.0,Iris-versicolor
62,5.9,3.0,4.2,1.5,Iris-versicolor
63,6.0,2.2,4.0,1.0,Iris-versicolor
64,6.1,2.9,4.7,1.4,Iris-versicolor
65,5.6,2.9,3.6,1.3,Iris-versicolor
66,6.7,3.1,4.4,1.4,Iris-versicolor
67,5.6,3.0,4.5,1.5,Iris-versicolor
68,5.8,2.7,4.1,1.0,Iris-versicolor
69,6.2,2.2,4.5,1.5,Iris-versicolor
70,5.6,2.5,3.9,1.1,Iris-versicolor
71,5.9,3.2,4.8,1.8,Iris-versicolor
72,6.1,2.8,4.0,1.3,Iris-versicolor
73,6.3,2.5,4.9,1.5,Iris-versicolor
74,6.1,2.8,4.7,1.2,Iris-versicolor
75,6.4,2.9,4.3,1.3,Iris-versicolor
76,6.6,3.0,4.4,1.4,Iris-versicolor
77,6.8,2.8,4.8,1.4,Iris-versicolor
78,6.7,3.0,5.0,1.7,Iris-versicolor
79,6.0,2.9,4.5,1.5,Iris-versicolor
80,5.7,2.6,3.5,1.0,Iris-versicolor
81,5.5,2.4,3.8,1.1,Iris-versicolor
82,5.5,2.4,3.7,1.0,Iris-versicolor
83,5.8,2.7,3.9,1.2,Iris-versicolor
84,6.0,2.7,5.1,1.6,Iris-versicolor
85,5.4,3.0,4.5,1.5,Iris-versicolor
86,6.0,3.4,4.5,1.6,Iris-versicolor
87,6.7,3.1,4.7,1.5,Iris-versicolor
88,6.3,2.3,4.4,1.3,Iris-versicolor
89,5.6,3.0,4.1,1.3,Iris-versicolor
90,5.5,2.5,4.0,1.3,Iris-versicolor
91,5.5,2.6,4.4,1.2,Iris-versicolor
92,6.1,3.0,4.6,1.4,Iris-versicolor
93,5.8,2.6,4.0,1.2,Iris-versicolor
94,5.0,2.3,3.3,1.0,Iris-versicolor
95,5.6,2.7,4.2,1.3,Iris-versicolor
96,5.7,3.0,4.2,1.2,Iris-versicolor
97,5.7,2.9,4.2,1.3,Iris-versicolor
98,6.2,2.9,4.3,1.3,Iris-versicolor
99,5.1,2.5,3.0,1.1,Iris-versicolor
100,5.7,2.8,4.1,1.3,Iris-versicolor
101,6.3,3.3,6.0,2.5,Iris-virginica
102,5.8,2.7,5.1,1.9,Iris-virginica
103,7.1,3.0,5.9,2.1,Iris-virginica
104,6.3,2.9,5.6,1.8,Iris-virginica
105,6.5,3.0,5.8,2.2,Iris-virginica
106,7.6,3.0,6.6,2.1,Iris-virginica
107,4.9,2.5,4.5,1.7,Iris-virginica
108,7.3,2.9,6.3,1.8,Iris-virginica
109,6.7,2.5,5.8,1.8,Iris-virginica
110,7.2,3.6,6.1,2.5,Iris-virginica
111,6.5,3.2,5.1,2.0,Iris-virginica
112,6.4,2.7,5.3,1.9,Iris-virginica
113,6.8,3.0,5.5,2.1,Iris-virginica
114,5.7,2.5,5.0,2.0,Iris-virginica
115,5.8,2.8,5.1,2.4,Iris-virginica
116,6.4,3.2,5.3,2.3,Iris-virginica
117,6.5,3.0,5.5,1.8,Iris-virginica
118,7.7,3.8,6.7,2.2,Iris-virginica
119,7.7,2.6,6.9,2.3,Iris-virginica
120,6.0,2.2,5.0,1.5,Iris-virginica
121,6.9,3.2,5.7,2.3,Iris-virginica
122,5.6,2.8,4.9,2.0,Iris-virginica
123,7.7,2.8,6.7,2.0,Iris-virginica
124,6.3,2.7,4.9,1.8,Iris-virginica
125,6.7,3.3,5.7,2.1,Iris-virginica
126,7.2,3.2,6.0,1.8,Iris-virginica
127,6.2,2.8,4.8,1.8,Iris-virginica
128,6.1,3.0,4.9,1.8,Iris-virginica
129,6.4,2.8,5.6,2.1,Iris-virginica
130,7.2,3.0,5.8,1.6,Iris-virginica
131,7.4,2.8,6.1,1.9,Iris-virginica
132,7.9,3.8,6.4,2.0,Iris-virginica
133,6.4,2.8,5.6,2.2,Iris-virginica
134,6.3,2.8,5.1,1.5,Iris-virginica
135,6.1,2.6,5.6,1.4,Iris-virginica
136,7.7,3.0,6.1,2.3,Iris-virginica
137,6.3,3.4,5.6,2.4,Iris-virginica
138,6.4,3.1,5.5,1.8,Iris-virginica
139,6.0,3.0,4.8,1.8,Iris-virginica
140,6.9,3.1,5.4,2.1,Iris-virginica
141,6.7,3.1,5.6,2.4,Iris-virginica
142,6.9,3.1,5.1,2.3,Iris-virginica
143,5.8,2.7,5.1,1.9,Iris-virginica
144,6.8,3.2,5.9,2.3,Iris-virginica
145,6.7,3.3,5.7,2.5,Iris-virginica
146,6.7,3.0,5.2,2.3,Iris-virginica
147,6.3,2.5,5.0,1.9,Iris-virginica
148,6.5,3.0,5.2,2.0,Iris-virginica
149,6.2,3.4,5.4,2.3,Iris-virginica
150,5.9,3.0,5.1,1.8,Iris-virginica
'''

In [ ]:
with open("iris.csv", "w") as f:
    f.write(IRIS_CSV.strip() + "\n")

# Blank out ~5 % of the four numeric columns (seeded -> the same gaps every run)
rng = np.random.default_rng(SEED)
lines = IRIS_CSV.strip().splitlines()
header, rows = lines[0], [line.split(",") for line in lines[1:]]
for row in rows:
    for col in (1, 2, 3, 4):
        if rng.random() < 0.05:
            row[col] = str(rng.choice(["", "NA", "?"]))
with open("iris_with_gaps.csv", "w") as f:
    f.write("\n".join([header] + [",".join(r) for r in rows]) + "\n")

print("columns:", header)
print("\n".join(lines[1:4]), "\n...")
print("examples of gaps:", [",".join(r) for r in rows if "" in r or "NA" in r or "?" in r][:3])

columns: Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa 
...
examples of gaps: ['7,4.6,3.4,1.4,NA,Iris-setosa', '13,4.8,3.0,,0.1,Iris-setosa', '17,5.4,3.9,1.3,NA,Iris-setosa']


## Step 1.5: Read the CSV with `io.load_csv`
Key options: `header=True` returns the column names, `usecols` picks columns, `dtype` controls the type, and `missing="fill"` (default) turns gaps into `NaN` while `missing="skip"` drops those rows.

*What to look for:* the shape is `(150, 4)` and `missing cells` is greater than 0; `missing="skip"` keeps fewer rows.

In [ ]:
# Numeric columns -> 2-D float array. Gaps (blank, NA, ?) become NaN.
X, names = io.load_csv("iris_with_gaps.csv", header=True, usecols=(1, 2, 3, 4))
print("columns :", names)
print("shape   :", X.shape, "| dtype:", X.dtype)
print("missing :", int(np.isnan(X).sum()), "cells are NaN")
print(X[:5])

# Option: drop every row that contains a gap
X_skip, _ = io.load_csv("iris_with_gaps.csv", header=True, usecols=(1, 2, 3, 4), missing="skip")
print("rows kept with missing='skip':", X_skip.shape[0], "of", X.shape[0])

# Text column -> array of strings (dtype handling)
species = io.load_csv("iris_with_gaps.csv", header=True, dtype=str, usecols=(5,))[0][:, 0]
print("species :", np.unique(species))

assert X.shape == (150, 4) and np.isnan(X).any() and X_skip.shape[0] < 150

columns : ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
shape   : (150, 4) | dtype: float64
missing : 27 cells are NaN
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
rows kept with missing='skip': 124 of 150
species : ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


# Part 2: Using your pipeline

A `Pipeline` chains steps: every step except the last must have `fit` and `transform`; the last may be a model with `predict`. Fitting learns from the **training** data only, and the same learned values (means, scales, ...) are then re-used on the **test** data. That avoids leaking test information into training.

## Step 2.1: A small model for the end of the pipeline
The toolkit provides the `Estimator` interface (a pipeline expects `fit` and `predict`) but not a model, so we write a tiny **nearest-centroid classifier** here. It uses `utils.pairwise_distances` for the distance computation.

In [ ]:
class NearestCentroid(pipeline.Estimator):
    """Predict the class whose feature average (centroid) is closest to the sample."""

    def fit(self, X, y=None):
        self.classes_ = np.unique(y)
        self.centroids_ = np.vstack([X[y == c].mean(axis=0) for c in self.classes_])   # one row per class
        return self

    def distances(self, X):
        return utils.pairwise_distances(X, self.centroids_, metric="euclidean")         # (n_samples, n_classes)

    def predict(self, X):
        return self.classes_[np.argmin(self.distances(X), axis=1)]

## Step 2.2: Split the data (70 % train / 30 % test)
A seeded random permutation makes the split reproducible without any ML library.

In [ ]:
order = np.random.default_rng(SEED).permutation(len(X))
cut = int(0.7 * len(X))
train, test = order[:cut], order[cut:]
X_train, y_train = X[train], species[train]
X_test, y_test = X[test], species[test]
print("train:", X_train.shape, "| test:", X_test.shape)
print("NaN cells in train/test:", int(np.isnan(X_train).sum()), "/", int(np.isnan(X_test).sum()),
      "(the pipeline will fill these)")

train: (105, 4) | test: (45, 4)
NaN cells in train/test: 20 / 7 (the pipeline will fill these)


## Step 2.3: Build, fit and use the pipeline
Steps are `(name, object)` pairs: `Imputer` fills the gaps with the column mean, `StandardScaler` rescales every feature to mean 0 / std 1, and the classifier predicts. One `fit` call fits all three in order.

*What to look for:* `steps` lists the three names, the predictions are species names, and no NaN had to be removed by hand.

In [ ]:
pipe = pipeline.Pipeline([
    ("impute", preprocessing.Imputer(strategy="mean")),
    ("scale", preprocessing.StandardScaler()),
    ("model", NearestCentroid()),
])

pipe.fit(X_train, y_train)               # learns means, scales and class centroids from TRAIN only
pred = pipe.predict(X_test)              # test data is transformed with the TRAIN statistics

print("steps       :", list(pipe.named_steps))
print("predictions :", pred[:6], "...")
print("truth       :", y_test[:6], "...")
print("train means used by the imputer:", np.round(pipe["impute"].statistics_, 3))

steps       : ['impute', 'scale', 'model']
predictions : ['Iris-setosa' 'Iris-versicolor' 'Iris-versicolor' 'Iris-virginica' 'Iris-setosa' 'Iris-virginica'] ...
truth       : ['Iris-setosa' 'Iris-versicolor' 'Iris-versicolor' 'Iris-virginica' 'Iris-setosa' 'Iris-virginica'] ...
train means used by the imputer: [5.886 3.021 3.889 1.242]


A pipeline can also be made only of transformers. `FeatureUnion` runs several transformers on the same data and places their outputs side by side, and `ColumnTransformer` treats numeric and categorical columns differently.

In [ ]:
union = pipeline.FeatureUnion([("standard", preprocessing.StandardScaler()),
                               ("minmax", preprocessing.MinMaxScaler())])
X_filled = preprocessing.Imputer().fit_transform(X)
print("FeatureUnion :", union.fit_transform(X_filled).shape, "= 4 standardised + 4 min-max columns")

mixed = np.column_stack([X_filled, species]).astype(object)          # 4 numeric columns + 1 text column
ct = preprocessing.ColumnTransformer(preprocessing.StandardScaler(), preprocessing.OneHotEncoder(),
                                     numeric_cols=[0, 1, 2, 3], categorical_cols=[4])
print("ColumnTransformer:", ct.fit_transform(mixed).shape, "= 4 scaled + 3 one-hot species columns")

FeatureUnion : (150, 8) = 4 standardised + 4 min-max columns
ColumnTransformer: (150, 7) = 4 scaled + 3 one-hot species columns


# Part 3: Reading the outputs

## 3A. Metrics (`metrics`)
How good are the predictions? Always compare **truth** (`y_test`) with **predictions** (`pred`).

| Metric | Read it as |
|---|---|
| `accuracy` | fraction of correct predictions |
| `precision` | of the flowers predicted as a class, how many really were that class |
| `recall` | of the flowers that really are a class, how many were found |
| `f1` | balance of precision and recall (low if either is low) |
| `confusion_matrix` | rows = true class, columns = predicted class; off-diagonal cells are mistakes |

In [ ]:
labels = np.unique(species)
print("accuracy :", round(metrics.accuracy(y_test, pred), 4))
for name, fn in (("precision", metrics.precision), ("recall", metrics.recall), ("f1", metrics.f1)):
    per_class = fn(y_test, pred, average=None, labels=labels)
    print(f"{name:9s}: macro = {fn(y_test, pred, average='macro'):.4f} | per class =",
          {str(k)[5:]: round(float(v), 3) for k, v in zip(labels, per_class)})

cm = metrics.confusion_matrix(y_test, pred, labels=labels)
print("\nconfusion matrix (rows = truth, columns = prediction)")
print("classes:", [str(l)[5:] for l in labels])
print(cm)
assert cm.sum() == len(y_test) and np.trace(cm) / cm.sum() == metrics.accuracy(y_test, pred)

accuracy : 0.8889
precision: macro = 0.8718 | per class = {'setosa': 1.0, 'versicolor': 0.692, 'virginica': 0.923}
recall   : macro = 0.8833 | per class = {'setosa': 1.0, 'versicolor': 0.9, 'virginica': 0.75}
f1       : macro = 0.8701 | per class = {'setosa': 1.0, 'versicolor': 0.783, 'virginica': 0.828}

confusion matrix (rows = truth, columns = prediction)
classes: ['setosa', 'versicolor', 'virginica']
[[19  0  0]
 [ 0  9  1]
 [ 0  4 12]]


**ROC and AUC** (binary problems). We keep the two hardest species (versicolor vs virginica) and score each flower by *how much closer it is to the virginica centroid*. The ROC curve plots the true-positive rate against the false-positive rate as the decision threshold moves; **AUC** is the area under it (0.5 = guessing, 1.0 = perfect). `mse` is the error measure for numeric predictions.

*What to look for:* AUC close to 1, and the same AUC from a second route (ranks, see 3B).

In [ ]:
prep = lambda A: pipe["scale"].transform(pipe["impute"].transform(A))      # same preprocessing as the pipeline
model = pipe["model"]
pair = np.isin(y_test, ["Iris-versicolor", "Iris-virginica"])
dist = model.distances(prep(X_test[pair]))
col = {c: k for k, c in enumerate(model.classes_)}
score = dist[:, col["Iris-versicolor"]] - dist[:, col["Iris-virginica"]]    # larger = more virginica-like
y_bin = (y_test[pair] == "Iris-virginica").astype(int)

fpr, tpr, thresholds = metrics.roc_curve(y_bin, score)
area = metrics.auc(fpr, tpr)
print("ROC points:", len(fpr), "| AUC =", round(area, 4), "| roc_auc_score =", round(metrics.roc_auc_score(y_bin, score), 4))
print("first points (FPR, TPR):", [(round(float(a), 2), round(float(b), 2)) for a, b in zip(fpr[:4], tpr[:4])])
print("mse example for numeric predictions ([1, 2, 3] vs [1, 2, 5]):", round(metrics.mse([1.0, 2.0, 3.0], [1.0, 2.0, 5.0]), 4))

ROC points: 27 | AUC = 0.9625 | roc_auc_score = 0.9625
first points (FPR, TPR): [(0.0, 0.0), (0.0, 0.06), (0.0, 0.12), (0.0, 0.19)]
mse example for numeric predictions ([1, 2, 3] vs [1, 2, 5]): 1.3333


## 3B. Ranks (`rank`)
Ranking replaces values by their position when sorted (1 = smallest). The interesting part is **ties**: what rank do two equal values get? `method` decides.

*What to look for:* the three `20`s tie for ranks 2, 3 and 4, and every tie rule gives them a different answer.

In [ ]:
demo = np.array([10, 20, 20, 30, 20])
print("values  :", demo)
for method in ("average", "min", "max", "dense", "ordinal"):
    print(f"{method:8s}:", rank.rank(demo, method=method))
print("descending (largest = rank 1):", rank.rank(demo, descending=True))
print("percentile rank (0..1)       :", rank.percentile_rank(demo))

values  : [10 20 20 30 20]
average : [1. 3. 3. 5. 3.]
min     : [1 2 2 5 2]
max     : [1 4 4 5 4]
dense   : [1 2 2 3 2]
ordinal : [1 2 3 5 4]
descending (largest = rank 1): [5. 3. 3. 1. 3.]
percentile rank (0..1)       : [0.  0.5 0.5 1.  0.5]


In [ ]:
# Percentiles of petal length (column 2); NaN gaps are ignored. rank.percentile uses the 0-100 scale
levels = [5, 25, 50, 75, 95]
petal = X[:, 2]
print("percentiles", levels, "->", rank.percentile(petal, levels))
assert np.allclose(rank.percentile(petal, levels), np.nanpercentile(petal, levels))

# Cross-check: AUC from the ranks of the scores (Mann-Whitney U) equals the ROC-curve AUC from 3A
r = rank.rank(score)                       # average ranks handle tied scores correctly
n_pos, n_neg = int(y_bin.sum()), int((1 - y_bin).sum())
auc_from_ranks = (r[y_bin == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
print("AUC via ranks:", round(float(auc_from_ranks), 4), "| AUC via ROC curve:", round(area, 4))
assert np.isclose(auc_from_ranks, area)

percentiles [5, 25, 50, 75, 95] -> [1.3 1.6 4.4 5.1 6. ]
AUC via ranks: 0.9625 | AUC via ROC curve: 0.9625


## 3C. Gradients (`optim`)
`grad` estimates the gradient of a scalar function and `jacobian` the matrix of derivatives of a vector function, using **finite differences** (no formulas needed).

* `central`: `(f(x+h) - f(x-h)) / 2h`, error shrinks like h², costs 2n calls.
* `forward`: `(f(x+h) - f(x)) / h`, error shrinks like h, costs n+1 calls.

*What to look for:* the numeric gradient matches the exact one to many digits; the step-size table shows the sweet spot (too large = truncation error, too small = round-off error).

In [ ]:
f = lambda x: x[0] ** 2 + 3 * np.sin(x[1]) + x[0] * x[1]                 # scalar function of 2 variables
exact = lambda x: np.array([2 * x[0] + x[1], 3 * np.cos(x[1]) + x[0]])    # its true gradient, for checking
x0 = np.array([1.5, -0.7])

g_central = optim.grad(f, x0)                          # default: central differences, h = 1e-5
g_forward = optim.grad(f, x0, method="forward")
print("exact gradient :", exact(x0))
print("central        :", g_central, "| max error", f"{np.abs(g_central - exact(x0)).max():.1e}")
print("forward        :", g_forward, "| max error", f"{np.abs(g_forward - exact(x0)).max():.1e}")
assert np.allclose(g_central, exact(x0), atol=1e-8)

F = lambda x: np.array([x[0] * x[1], x[0] ** 2 + x[1], np.sin(x[0])])    # vector function: 2 inputs -> 3 outputs
J_exact = lambda x: np.array([[x[1], x[0]], [2 * x[0], 1.0], [np.cos(x[0]), 0.0]])
J = optim.jacobian(F, x0)
print("\nJacobian (rows = outputs, columns = inputs):\n", J)
assert J.shape == (3, 2) and np.allclose(J, J_exact(x0), atol=1e-8)

exact gradient : [2.3    3.7945]
central        : [2.3    3.7945] | max error 4.7e-11
forward        : [2.3    3.7945] | max error 1.0e-05

Jacobian (rows = outputs, columns = inputs):
 [[-0.7     1.5   ]
 [ 3.      1.    ]
 [ 0.0707  0.    ]]


In [ ]:
print(f"{'step h':>8} | {'central error':>13} | {'forward error':>13}")
for h in (1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11):
    ec = np.abs(optim.grad(f, x0, h=h) - exact(x0)).max()
    ef = np.abs(optim.grad(f, x0, h=h, method="forward") - exact(x0)).max()
    print(f"{h:8.0e} | {ec:13.2e} | {ef:13.2e}")

  step h | central error | forward error
   1e-01 |      3.82e-03 |      1.00e-01
   1e-03 |      3.82e-07 |      1.00e-03
   1e-05 |      4.68e-11 |      1.00e-05
   1e-07 |      1.11e-09 |      9.93e-08
   1e-09 |      3.01e-07 |      1.90e-07
   1e-11 |      6.47e-06 |      1.76e-05


# Part 4: the rest of the toolkit
## 4A. Statistics (`stats`)
`describe` summarises each column and **ignores NaN**, reporting how many values were missing.

In [ ]:
print(stats.format_summary(stats.describe(X, names=names)))

counts, edges = stats.histogram(petal, bins=8)
print("\nhistogram of petal length")
print(stats.format_histogram(counts, edges, width=30))
print("quartiles:", stats.quantile(petal, [0.25, 0.5, 0.75]), "| median:", stats.median(petal))
label, freq = stats.mode(species)
print("most common species:", str(label), f"({freq} rows)")
assert np.allclose(stats.mean(X, axis=0), np.nanmean(X, axis=0))
print("\nvariance per column  :", stats.var(X, axis=0))
rs = stats.RunningStats()                                          # Welford: one chunk at a time
for chunk, _ in io.iter_csv("iris_with_gaps.csv", chunk_size=50, header=True, usecols=(1, 2, 3, 4)):
    rs.update(chunk)
print("Welford, 50 rows at a time -> mean:", rs.mean(), "| var:", rs.var())
assert np.allclose(rs.mean(), stats.mean(X, axis=0)) and np.allclose(rs.var(), stats.var(X, axis=0))

stat     SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
count              144           145            142           142
missing              6             5              8             8
mean             5.833         3.066          3.777         1.227
std              0.830         0.431          1.741         0.759
min              4.300         2.000          1.000         0.100
25%              5.100         2.800          1.600         0.300
50%              5.800         3.000          4.400         1.300
75%              6.400         3.300          5.100         1.800
max              7.900         4.400          6.900         2.500

histogram of petal length
[   1.000,    1.738) | ############################## 44
[   1.738,    2.475) | # 2
[   2.475,    3.213) | # 1
[   3.213,    3.950) | ####### 10
[   3.950,    4.688) | ################### 28
[   4.688,    5.425) | ##################### 31
[   5.425,    6.163) | ############## 20
[   6.163,    6.900] | #### 6
qua

## 4B. Sorting, top-k, quickselect, binary search (`sort_search`)
Equal values keep their original order (stable), `NaN` is treated as missing, and top-k avoids a full sort.

In [ ]:
petal_full = preprocessing.Imputer().fit_transform(X)[:, 2]     # gaps filled with the column mean, so its median differs slightly from 4A
print("stable_sort of [3, 1, 3, 1] (indices):", sort_search.stable_sort([3, 1, 3, 1], return_indices=True))

idx = sort_search.topk(petal_full, 5)
print("5 longest petals:", petal_full[idx], "-> rows", idx)
print("3 shortest      :", sort_search.topk(petal_full, 3, largest=False, return_indices=False))

mid = len(petal_full) // 2
print("quickselect median:", sort_search.quickselect(petal_full, mid))
assert sort_search.quickselect(petal_full, mid) == np.sort(petal_full)[mid]

petal_sorted = sort_search.stable_sort(petal_full)                 # binary_search needs sorted input
pos, found = sort_search.binary_search(petal_sorted, 1.5)
print("binary_search(1.5) on sorted petals -> index", pos, "| found:", found, "| value:", petal_sorted[pos])
assert pos == np.searchsorted(petal_sorted, 1.5)

with warnings.catch_warnings(record=True) as caught:       # NaN is ignored, with a warning
    warnings.simplefilter("always")
    sort_search.topk(np.array([3.0, np.nan, 7.0]), 2)
print("NaN warning:", caught[0].message)

stable_sort of [3, 1, 3, 1] (indices): [1 3 0 2]
5 longest petals: [6.9 6.7 6.7 6.6 6.4] -> rows [118 117 122 105 131]
3 shortest      : [1.  1.1 1.2]
quickselect median: 4.2
binary_search(1.5) on sorted petals -> index 21 | found: True | value: 1.5
NaN warning: topk: ignoring 1 NaN value(s) out of 3


## 4C. Utilities (`utils`)
Stable activations never overflow, and distances/top-k/batching work on whole arrays.

In [ ]:
logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    print("naive softmax :", np.exp(logits) / np.exp(logits).sum(), " <- overflow")
print("stable softmax:", utils.softmax(logits))
print("logsumexp     :", utils.logsumexp(logits), "| sigmoid(-1000, 0, 1000):", utils.sigmoid(np.array([-1000.0, 0.0, 1000.0])))

Z = prep(X)
for metric in ("euclidean", "manhattan", "cosine"):
    print(f"{metric:9s} distance of row 0 to rows 1-3:", utils.pairwise_distances(Z[:1], Z[1:4], metric=metric)[0])

values, positions = utils.topk_along_axis(Z, 2, axis=0)          # top-2 per column
print("top-2 per feature:\n", values)
print("batch sizes:", [Z[sl].shape[0] for sl in utils.batch_slices(len(Z), 64)])

naive softmax : [nan nan nan]  <- overflow
stable softmax: [0.09   0.2447 0.6652]
logsumexp     : 1002.4076059644444 | sigmoid(-1000, 0, 1000): [0.  0.5 1. ]
euclidean distance of row 0 to rows 1-3: [1.3326 0.9252 1.2118]
manhattan distance of row 0 to rows 1-3: [1.5524 1.3307 1.7138]
cosine    distance of row 0 to rows 1-3: [0.1361 0.062  0.1076]
top-2 per feature:
 [[2.436  3.6142 1.8249 1.774 ]
 [2.1941 3.09   1.7037 1.774 ]]
batch sizes: [64, 64, 22]


## 4D. Edge cases and error messages
Empty input, all-equal values, ties, NaN, extreme `k`, non-contiguous arrays, and clear errors.

In [ ]:
print("empty array, top-k          :", utils.topk_along_axis(np.array([]), 3)[0], "| k larger than n:", utils.topk_along_axis([5, 1, 3], 99)[0])
print("all-equal values, ranks     :", rank.rank([7, 7, 7, 7]))
print("ties in top-k (earliest win):", sort_search.topk(np.array([2, 2, 2, 1]), 2))
print("strided (non-contiguous)    :", utils.topk_along_axis(np.arange(12)[::3], 2)[0])
print("NaN sorts last              :", sort_search.stable_sort([3.0, np.nan, 1.0]))
print("logsumexp([-inf, -inf])     :", utils.logsumexp([-np.inf, -np.inf]))


def show(label, fn):
    try:
        fn()
    except Exception as exc:                                    # show the message a user would see
        print(f"{label:26s} -> {type(exc).__name__}: {exc}")


show("rank with NaN", lambda: rank.rank([1.0, np.nan]))
show("transform before fit", lambda: preprocessing.StandardScaler().transform([[1.0]]))
show("wrong feature count", lambda: preprocessing.StandardScaler().fit([[1.0, 2.0]]).transform([[1.0]]))
show("length mismatch", lambda: metrics.accuracy([0, 1], [0]))
show("function not callable", lambda: optim.grad(3, [1.0]))
show("missing file", lambda: io.load_csv("no_such_file.csv"))

empty array, top-k          : [] | k larger than n: [5 3 1]
all-equal values, ranks     : [2.5 2.5 2.5 2.5]
ties in top-k (earliest win): [0 1]
strided (non-contiguous)    : [9 6]
NaN sorts last              : [ 1.  3. nan]
logsumexp([-inf, -inf])     : -inf
rank with NaN              -> ValueError: x contains NaN; remove or replace it first
transform before fit       -> RuntimeError: StandardScaler is not fitted; call fit/partial_fit first
wrong feature count        -> ValueError: Expected 2 features, got 1
length mismatch            -> ValueError: y_true and y_pred must have the same length, got 2 and 1
function not callable      -> TypeError: f must be callable, got int
missing file               -> FileNotFoundError: No such file: 'no_such_file.csv'


# Part 5: benchmark, vectorised vs Python loops
The repository's `benchmarks/run_benchmarks.py` times each NumCompute function against a plain-Python loop on identical, seeded data and checks that both give the same answer. The full run takes about 15 seconds.

*What to look for:* the `speed-up` column (above 1x means the vectorised version is faster) and the environment notes above the table. Very small inputs can show less than 1x because NumPy has a fixed start-up cost per call, and the speed-up grows with size.

In [ ]:
bench_dir = None
for base, dirs, files in os.walk(CLONE_DIR):
    if "run_benchmarks.py" in files:
        bench_dir = base
        break

if bench_dir is None:
    print("benchmarks/run_benchmarks.py not found in the downloaded repository, skipping.")
else:
    out = subprocess.run([sys.executable, "run_benchmarks.py", "--repeat", "3"],
                         cwd=bench_dir, capture_output=True, text=True)
    text = out.stdout
    print(text[text.find("**Environment**"):] if "**Environment**" in text else text[-3000:])
    if out.returncode != 0:
        print(out.stderr[-1500:])

**Environment**
- python: 3.13.16
- numpy: 2.1.3
- platform: Linux-6.6.122+-x86_64-with-glibc2.39
- processor: x86_64
- cpu_count: 2
- seed: 0

| case | n | baseline | loop (ms) | vectorised (ms) | speed-up | results match |
|---|---:|---|---:|---:|---:|:---:|
| StandardScaler | 1,000 | python loops | 2.702 | 0.463 | 5.8x | yes |
| StandardScaler | 10,000 | python loops | 33.401 | 2.675 | 12.5x | yes |
| StandardScaler | 100,000 | python loops | 388.384 | 25.188 | 15.4x | yes |
| MinMaxScaler | 1,000 | python loops | 1.952 | 0.406 | 4.8x | yes |
| MinMaxScaler | 10,000 | python loops | 23.075 | 2.384 | 9.7x | yes |
| MinMaxScaler | 100,000 | python loops | 295.419 | 27.731 | 10.7x | yes |
| OneHotEncoder | 1,000 | dict + loop | 0.341 | 0.929 | 0.4x | yes |
| OneHotEncoder | 10,000 | dict + loop | 3.262 | 6.600 | 0.5x | yes |
| OneHotEncoder | 100,000 | dict + loop | 41.422 | 65.936 | 0.6x | yes |
| rank (average, ties) | 1,000 | sort + tie loop | 0.489 | 0.379 | 1.3x | yes |
| rank (av

# Wrap-up
In this notebook NumCompute was installed from GitHub and used end to end:

- **io**: read an Iris CSV with gaps (`""`, `NA`, `?`), filling or skipping them, and reading it in chunks
- **preprocessing + pipeline**: `Imputer -> StandardScaler -> classifier` in one `Pipeline`, plus `FeatureUnion` and `ColumnTransformer`
- **outputs**: metrics (accuracy, precision/recall/F1, confusion matrix, ROC/AUC), ranks (ties, percentiles) and gradients (finite differences, Jacobian)
- **stats, sort_search, utils**: summaries, histogram, quantiles, streaming Welford statistics, top-k, quickselect, binary search, stable softmax
- **benchmark**: vectorised code against plain-Python loops, with every result checked for a match

See `README.md` for installation, the API overview, and how to run the tests and benchmarks.